# 量比
接下來這一個版本的程式碼是要用量比來計算就是這一檔股票截至該時間點的累積成交金額/過去20日的平均成交金額。


過去20日平均成交金額：`C:\Users\user\Documents\_15_預估量\exp_vol_project\data\stock_volume.parquet`
1minK資料：`C:\Users\user\Documents\_04_lupFinmind\1min\{y-m-d}\{stock_id}.parquet`

該時間點的累積成交金額計算方式，使用一分K資料的((開盤價+收盤價)/2 * 成交量 * 1000)，將其累加起來進行計算。



In [ ]:
def plot_stock_detail(stock_id, date):
    """輸入股票代碼與日期，畫出走勢圖 + 預估成交金額走勢"""
    date_compact = date.replace('-', '')

    # 載入 1min K
    kbar_path = os.path.join(KBAR_1M_DIR, date, f'{stock_id}.parquet')
    if not os.path.exists(kbar_path):
        print(f'找不到 {stock_id} 在 {date} 的 1min K 資料')
        return
    df_k = pd.read_parquet(kbar_path)
    df_k['datetime'] = pd.to_datetime(date + ' ' + df_k['minute'])
    df_k = df_k.set_index('datetime')

    # 載入 vol_exp
    vol_path = os.path.join(VOL_EXP_DIR, f'vol_exp_{date_compact}.parquet')
    if not os.path.exists(vol_path):
        print(f'找不到 {date} 的 vol_exp 資料')
        return
    df_ve = pd.read_parquet(vol_path)
    if stock_id not in df_ve.columns:
        print(f'{stock_id} 不在 vol_exp 中')
        return
    vol_series = df_ve[stock_id]

    # 對齊時間
    common_idx = df_k.index.intersection(vol_series.index)
    high = df_k.loc[common_idx, 'high']
    vol_exp = vol_series.loc[common_idx]
    est_turnover = high * vol_exp * 1000  # 預估成交金額(元)

    # 查 avg_turnover_20d
    df_sv = pd.read_parquet(STOCK_VOL_PATH)
    df_sv['date_str'] = df_sv['date'].astype(str).str[:10]
    row = df_sv[(df_sv['date_str'] == date) & (df_sv['stock_id'] == stock_id)]
    avg_20d = row['avg_turnover_20d'].values[0] if not row.empty else None

    # 畫圖
    fig = make_subplots(
        rows=2, cols=1, shared_xaxes=True,
        subplot_titles=(f'{stock_id} 走勢圖', f'{stock_id} 預估成交金額'),
        vertical_spacing=0.08, row_heights=[0.5, 0.5]
    )

    # 上圖: 收盤價走勢
    fig.add_trace(go.Scatter(
        x=common_idx, y=df_k.loc[common_idx, 'close'].values,
        mode='lines', name='收盤價',
        line=dict(width=1.5, color='#1f77b4')
    ), row=1, col=1)

    # 下圖: 預估成交金額
    fig.add_trace(go.Scatter(
        x=common_idx, y=est_turnover.values / 1e8,
        mode='lines', name='預估成交金額(億)',
        line=dict(width=1.5, color='#ff7f0e')
    ), row=2, col=1)

    # 20日均量參考線
    if avg_20d is not None:
        fig.add_hline(
            y=avg_20d / 1e8, row=2, col=1,
            line=dict(color='white', width=1, dash='dash'),
            annotation_text=f'20日均量 {avg_20d/1e8:.1f}億',
            annotation_position='top left', annotation_font_size=10
        )

    fig.update_layout(
        height=700,
        title_text=f'{stock_id} - {date} 日內分析',
        template='plotly_dark', showlegend=False
    )
    fig.update_xaxes(tickformat='%H:%M')
    fig.update_yaxes(title_text='價格', row=1, col=1)
    fig.update_yaxes(title_text='預估成交金額(億)', row=2, col=1)
    fig.show()

# === 使用方式 ===
plot_stock_detail('2337', '2026-02-04')